## Use SHAP to map mortality drivers

Fit a model to all data, then use SHAP to derive explanations.

In [ ]:
import xarray as xr
import rioxarray
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import lightgbm as lgb
import fasttreeshap
import warnings
import logging
from tqdm.autonotebook import tqdm

import const
from gbm import split_xy, make_zif_quantile_estimator, get_results, balance_zeros, safe_logit

In [ ]:
warnings.filterwarnings("ignore", message="X does not have valid feature names")
logging.getLogger().setLevel(logging.CRITICAL)

In [ ]:
westmort = xr.open_zarr("../data_working/westmort.zarr/").compute().rio.write_crs(const.PROJECTION)
westmort

In [ ]:
def make_data_frame(agent: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    agent_ba = f"{agent}_ba"
    agent_mort = f"{agent}_mort"
    agent_target = f"{agent}_target"
    
    cols_to_select = const.GBM_COVARIATES["hydro"] +\
        const.GBM_COVARIATES["topo"] +\
        const.GBM_COVARIATES["climate"] +\
        const.GBM_COVARIATES["structure"]+\
        [agent_ba, agent_mort, agent_target]

    westmort_df = westmort[cols_to_select].to_dataframe()
    westmort_df[agent_mort] = safe_logit(westmort_df[agent_mort])
    westmort_df = westmort_df[westmort_df[agent_ba] > 0].dropna()

    return westmort_df

In [ ]:
results = {}
agents = list(const.HOST_DCA_CODES.keys())
gbm_args = {
    "learning_rate": 0.1,
    "n_estimators": 100,
    "num_leaves": 16,
    "bagging_fraction": 0.8,
    "feature_fraction": 0.8
}

for agent in tqdm(agents):
    target_var = f"{agent}_target"
    
    df = make_data_frame(agent)
    # print(train.columns)
    model = make_zif_quantile_estimator(gbm_args, gbm_args) # same args for classifier and regressor

    df_subsample = balance_zeros(df, target_var)
    
    X_subsample, y_subsample = split_xy(df_subsample, target_var)
    X, y = split_xy(df, target_var)

    model.fit(X_subsample, y_subsample)
    y_hat = model.predict(X)
    
    model_perf = get_results(y, y_hat)

    # Here we are specifically looking at the logits of the regressor arm.
    # The property regressor_ gives us the fitted object, while regressor
    # (without the underscore) gives us the unfitted object.
    explainer = fasttreeshap.TreeExplainer(model.regressor_.regressor_, algorithm="auto")
    shap_values = pd.DataFrame(
        data=explainer(X).values,
        columns=X.columns,
        index=X.index
    )

    results[agent] = {
        "model": model,
        "X": X,
        "y": y,
        "performance": model_perf,
        "shap": shap_values
    }

## Build xarray object of SHAP values

SHAP gives us a data frame only at indices where we have nonzero host BA and no otherwise missing values. To use xarray for mapping, we have to convert this back to an xarray dataset. This doesn't work with a naive `to_xarray()` because not all values of x/y coordinates in the original data were present for modeling. First, we build a "template" DF with all the indices from the full xarray dataset, join onto that, and *then* convert to xarray. This eats a ton of RAM.

In [ ]:
template_df = westmort["def"].to_dataframe().drop(columns=["spatial_ref", "def"])

In [ ]:
shap_xarray_objs = [
    template_df.join(results[agent]["shap"], how="left").to_xarray()\
        .rename({f"{agent}_ba":"agent_ba", f"{agent}_mort":"agent_mort"})\
        .to_dataarray()\
        .assign_coords(agent=agent)
    for agent in results
]

In [ ]:
shap_xarray = xr.concat(shap_xarray_objs, "agent")

In [ ]:
shap_xarray.to_zarr("../data_working/gbm_shap.zarr")

In [ ]:
# del shap_xarray
# del shap_xarray_objs

In [ ]:
pd.json_normalize([results[key]["performance"] for key in results])